# JAXA catalog explorer

Browse the bundled JAXA catalog without touching the network. The catalog ships 118 STAC collections (the `jaxa-earth` protocol) plus six strategic G-Portal mission products (the `gportal` protocol), all driven by a `protocol` discriminator on each row.

> No credentials needed; no `[jaxa]` extra needed for this notebook either.

## Open the catalog

In [ ]:
from earthlens.jaxa import Catalog

cat = Catalog()
len(cat)

## Split by protocol

The two protocols are routed independently by the backend. `by_protocol(...)` lists the canonical keys for each.

In [ ]:
jaxa_earth_keys = cat.by_protocol('jaxa-earth')
gportal_keys = cat.by_protocol('gportal')
print(f'jaxa-earth: {len(jaxa_earth_keys)} collections')
print(f'gportal:    {len(gportal_keys)} products')
print()
print('First 5 jaxa-earth keys:')
for k in jaxa_earth_keys[:5]:
    print(f'  {k}')
print()
print('Every gportal key:')
for k in gportal_keys:
    print(f'  {k}')

## Friendly aliases

The most popular collections have short, friendly aliases — `elevation`, `gsmap`, `palsar2`, etc. — that resolve to the canonical row through `cat.get(alias)` or via the `EarthLens(..., variables=[alias])` facade.

In [ ]:
for alias in ['elevation', 'gsmap', 'palsar2', 'earthcare', 'gosat-gw']:
    row = cat.get(alias)
    print(f'{alias:12s} -> {row.key:32s} ({row.protocol})')

## Inspect a row

Every row carries its protocol-specific identifier — a `collection` (STAC name) for `jaxa-earth` or a `short_name` (numeric id) for `gportal` — plus a default band where applicable.

In [ ]:
aw3d30 = cat.get('elevation')
print('canonical key:', aw3d30.key)
print('protocol:     ', aw3d30.protocol)
print('collection:   ', aw3d30.collection)
print('default band: ', aw3d30.default_band)
print()
palsar2 = cat.get('palsar2')
print('canonical key:', palsar2.key)
print('protocol:     ', palsar2.protocol)
print('short_name:   ', palsar2.short_name)
print('description:  ', palsar2.description)

## Unknown keys raise with a did-you-mean hint

The catalog uses `difflib.get_close_matches` to suggest the nearest known key on a typo.

In [ ]:
try:
    cat.get('aw3d3')
except ValueError as exc:
    print(exc)

## Refreshing against the live SDK universes

The CLI command `earthlens datasets refresh jaxa` walks `jaxa.earth.ImageCollectionList` (STAC) and `gportal.datasets()` (G-Portal) to diff the bundled YAML against the live IDs. Both SDKs are optional — the `[jaxa]` extra installs them.

```bash
earthlens datasets refresh jaxa
```